# Images from the YAOKI rover camera

Load and display reconstructed YAOKI camera raw image packets from `../data/reconstructed_images`.


In [ ]:
from pathlib import Path
import math
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import color, exposure
import cv2
import os
import botocore
import boto3

IMAGE_DIR = Path('../data/reconstructed_images')
IMAGE_WIDTH = 320
IMAGE_HEIGHT = 240
PACKET_SIZE_BYTES = 168
PACKET_HEADER_BYTES = 8
PACKET_PAYLOAD_BYTES = PACKET_SIZE_BYTES - PACKET_HEADER_BYTES
IMAGE_TOTAL_PACKETS = 960
RAW_MAGIC = b'\x89\x89\xaa\xaa'

plt.rcParams['figure.dpi'] = 120

In [ ]:
# Connect to the S3 server
BUCKET = "im2-yaoki-rover"
REGION = "ap-northeast-1"

s3 = boto3.client("s3", region_name=REGION, config=botocore.config.Config(signature_version=botocore.UNSIGNED))

# Download directory for the timeseries data for YAOKI.
PREFIX = "images/"
LOCAL_DIR = os.path.join(".", "images")

In [ ]:
# Optionally list all files on the S3.
# response = s3.list_objects_v2(Bucket=BUCKET)
# for obj in response["Contents"]:
#     print(obj["Key"], obj["Size"])

In [ ]:
# Download all the timeseries data for YAOKI.  
# Once the data is downloaded you can comment this cell out.

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/"):   # skip directory placeholder objects
            continue
        local_path = os.path.join(LOCAL_DIR, os.path.relpath(key, PREFIX))
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(BUCKET, key, local_path)
        print(f"Downloaded: {key}")

print("Done")

## Raw Image Loader

The `.raw` files are concatenated camera packets, not DSLR-style RAW files, so tools such as `rawpy`/LibRaw do not know how to decode them. Each packet has an 8-byte header followed by 160 bytes of image payload. The header begins with the byte sequence `89 89 aa aa`, followed by a 16-bit image number and a 16-bit packet number. A complete image contains 960 packets.

<svg width='760' height='190' viewBox='0 0 760 190' xmlns='http://www.w3.org/2000/svg' role='img' aria-label='YAOKI raw packet layout'>
  <style>
    .label { font: 13px sans-serif; fill: #222; }
    .small { font: 11px sans-serif; fill: #444; }
    .byte { font: 11px monospace; fill: #1f2933; }
  </style>
  <text x='20' y='24' class='label'>One 168-byte camera packet</text>
  <rect x='20' y='45' width='160' height='58' fill='#d8ecff' stroke='#1b5d8f'/>
  <rect x='180' y='45' width='80' height='58' fill='#e6f4d7' stroke='#4d7c2f'/>
  <rect x='260' y='45' width='80' height='58' fill='#fff0c2' stroke='#9b6b00'/>
  <rect x='340' y='45' width='380' height='58' fill='#f5d6d6' stroke='#9d3b3b'/>
  <text x='53' y='72' class='byte'>89 89 aa aa</text>
  <text x='70' y='91' class='small'>magic, 4 bytes</text>
  <text x='193' y='72' class='byte'>00 05</text>
  <text x='191' y='91' class='small'>image #</text>
  <text x='273' y='72' class='byte'>00 09</text>
  <text x='270' y='91' class='small'>packet #</text>
  <text x='475' y='72' class='label'>160-byte UYVY payload</text>
  <text x='450' y='91' class='small'>80 pixels, 2 bytes per pixel</text>
  <line x1='20' y1='118' x2='720' y2='118' stroke='#333'/>
  <line x1='20' y1='113' x2='20' y2='123' stroke='#333'/>
  <line x1='340' y1='113' x2='340' y2='123' stroke='#333'/>
  <line x1='720' y1='113' x2='720' y2='123' stroke='#333'/>
  <text x='134' y='139' class='small'>8-byte header</text>
  <text x='506' y='139' class='small'>160-byte payload</text>
  <text x='20' y='170' class='small'>960 packets x 160 payload bytes = 153600 image bytes = 320 x 240 x 2</text>
</svg>

The payload is UYVY, also called YUV 4:2:2. Bytes are grouped in fours as `U0 Y0 V0 Y1`. The two neighboring pixels have separate luma/brightness samples (`Y0`, `Y1`) but share one chroma pair (`U0`, `V0`). This gives 2 bytes per pixel: `320 * 240 * 2 = 153600` bytes, which is exactly `960 * 160` packet payload bytes. Missing packets are filled as black for display, and the loader also returns a validity mask so later analysis can ignore those pixels explicitly.

<svg width='760' height='230' viewBox='0 0 760 230' xmlns='http://www.w3.org/2000/svg' role='img' aria-label='UYVY byte layout'>
  <style>
    .label { font: 13px sans-serif; fill: #222; }
    .small { font: 11px sans-serif; fill: #444; }
    .byte { font: 12px monospace; fill: #1f2933; }
  </style>
  <text x='20' y='24' class='label'>UYVY / YUV 4:2:2 byte group</text>
  <rect x='40' y='48' width='120' height='54' fill='#d8ecff' stroke='#1b5d8f'/>
  <rect x='160' y='48' width='120' height='54' fill='#f0f0f0' stroke='#555'/>
  <rect x='280' y='48' width='120' height='54' fill='#ffdede' stroke='#9d3b3b'/>
  <rect x='400' y='48' width='120' height='54' fill='#f0f0f0' stroke='#555'/>
  <text x='88' y='79' class='byte'>U0</text>
  <text x='208' y='79' class='byte'>Y0</text>
  <text x='328' y='79' class='byte'>V0</text>
  <text x='448' y='79' class='byte'>Y1</text>
  <text x='77' y='119' class='small'>shared blue-difference chroma</text>
  <text x='181' y='139' class='small'>pixel 0 brightness</text>
  <text x='312' y='119' class='small'>shared red-difference chroma</text>
  <text x='421' y='139' class='small'>pixel 1 brightness</text>
  <rect x='145' y='170' width='110' height='38' fill='#f7f7f7' stroke='#666'/>
  <rect x='305' y='170' width='110' height='38' fill='#f7f7f7' stroke='#666'/>
  <text x='178' y='194' class='label'>Pixel 0</text>
  <text x='338' y='194' class='label'>Pixel 1</text>
  <path d='M100 102 C100 150 178 150 190 170' fill='none' stroke='#1b5d8f' stroke-width='2'/>
  <path d='M220 102 C220 135 207 145 200 170' fill='none' stroke='#555' stroke-width='2'/>
  <path d='M340 102 C340 150 222 150 210 170' fill='none' stroke='#9d3b3b' stroke-width='2'/>
  <path d='M100 102 C100 155 338 155 350 170' fill='none' stroke='#1b5d8f' stroke-width='2' stroke-dasharray='5 4'/>
  <path d='M340 102 C340 155 382 155 370 170' fill='none' stroke='#9d3b3b' stroke-width='2' stroke-dasharray='5 4'/>
  <path d='M460 102 C460 135 387 145 360 170' fill='none' stroke='#555' stroke-width='2'/>
  <text x='545' y='72' class='small'>Four bytes encode two pixels.</text>
  <text x='545' y='93' class='small'>Luma is sampled for every pixel.</text>
  <text x='545' y='114' class='small'>Chroma is shared by each pair.</text>
</svg>

In [ ]:
def uyvy_to_rgb(payload, width=IMAGE_WIDTH, height=IMAGE_HEIGHT):
    expected_size = width * height * 2
    if len(payload) != expected_size:
        raise ValueError(f'Expected {expected_size} payload bytes, got {len(payload)}')

    uyvy = np.frombuffer(payload, dtype=np.uint8).reshape(height, width, 2)
    return cv2.cvtColor(uyvy, cv2.COLOR_YUV2RGB_UYVY)


def packet_mask_to_pixel_mask(packet_mask, width=IMAGE_WIDTH, height=IMAGE_HEIGHT):
    pixels_per_packet = PACKET_PAYLOAD_BYTES // 2
    expected_pixels = width * height
    if len(packet_mask) * pixels_per_packet != expected_pixels:
        raise ValueError('Packet geometry does not match image geometry')
    return np.repeat(packet_mask, pixels_per_packet).reshape(height, width)


def load_raw_image(raw_path, width=IMAGE_WIDTH, height=IMAGE_HEIGHT, total_packets=IMAGE_TOTAL_PACKETS):
    raw_path = Path(raw_path)
    data = raw_path.read_bytes()
    if len(data) % PACKET_SIZE_BYTES != 0:
        raise ValueError(f'{raw_path.name} size is not a multiple of {PACKET_SIZE_BYTES} bytes')

    payload = bytearray([128, 0, 128, 0] * (width * height // 2))
    packet_mask = np.zeros(total_packets, dtype=bool)
    packet_numbers = []
    image_numbers = []

    for offset in range(0, len(data), PACKET_SIZE_BYTES):
        packet = data[offset:offset + PACKET_SIZE_BYTES]
        if packet[:4] != RAW_MAGIC:
            raise ValueError(f'{raw_path.name} has invalid packet magic at byte offset {offset}')

        image_number, packet_number = struct.unpack('>HH', packet[4:8])
        image_numbers.append(image_number)
        packet_numbers.append(packet_number)

        if packet_number >= total_packets:
            continue

        payload_start = packet_number * PACKET_PAYLOAD_BYTES
        payload[payload_start:payload_start + PACKET_PAYLOAD_BYTES] = packet[PACKET_HEADER_BYTES:]
        packet_mask[packet_number] = True

    received_packets = int(packet_mask.sum())
    missing_packets = total_packets - received_packets

    return {
        'path': raw_path,
        'image': uyvy_to_rgb(payload, width=width, height=height),
        'valid_mask': packet_mask_to_pixel_mask(packet_mask, width=width, height=height),
        'packet_mask': packet_mask,
        'image_numbers': sorted(set(image_numbers)),
        'received_packets': received_packets,
        'missing_packets': missing_packets,
        'first_packet': min(packet_numbers) if packet_numbers else None,
        'last_packet': max(packet_numbers) if packet_numbers else None,
    }

## Raw File Inventory

In [ ]:
raw_files = sorted(IMAGE_DIR.glob('*.raw'))
print(f'Found {len(raw_files)} raw image files in {IMAGE_DIR}')

records = []
raw_images = []
for raw_file in raw_files:
    loaded = load_raw_image(raw_file)
    raw_images.append(loaded)
    records.append({
        'file': raw_file.name,
        'image_numbers': loaded['image_numbers'],
        'received_packets': loaded['received_packets'],
        'missing_packets': loaded['missing_packets'],
        'first_packet': loaded['first_packet'],
        'last_packet': loaded['last_packet'],
        'size_bytes': raw_file.stat().st_size,
    })

raw_df = pd.DataFrame(records)
raw_df

## Display One Raw Image

In [ ]:
image_index = 0
loaded = raw_images[image_index]

plt.figure(figsize=(6, 4.5))
plt.imshow(loaded['image'])
plt.title(
    f"{loaded['path'].name} | "
    f"packets {loaded['received_packets']}/{IMAGE_TOTAL_PACKETS} | "
    f"missing {loaded['missing_packets']}"
)
plt.axis('off')
plt.show()

## Display All Raw Images

In [ ]:
columns = 4
rows = math.ceil(len(raw_images) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(columns * 3.2, rows * 2.7), squeeze=False)

for ax, loaded in zip(axes.flat, raw_images):
    ax.imshow(loaded['image'])
    ax.set_title(
        f"{loaded['path'].stem}\n"
        f"{loaded['received_packets']}/{IMAGE_TOTAL_PACKETS} packets",
        fontsize=8,
    )
    ax.axis('off')

for ax in axes.flat[len(raw_images):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Display masked mean and standard deviation images

In [ ]:
# Notice the packet loss shows up as black pixels. We want a mean image and a
# standard-deviation image that exclude missing packet samples at each pixel.

# First let's put all images in a stack.
image_stack = np.stack([loaded['image'] for loaded in raw_images], axis=0)
valid_stack = np.stack([loaded['valid_mask'] for loaded in raw_images], axis=0)
print(f'Image stack shape: {image_stack.shape}')

# scikit-image handles RGB-to-grayscale conversion using the standard luminance weights.
gray_stack = color.rgb2gray(image_stack)

# In NumPy masked arrays, mask=True means "exclude this value from calculations".
# Use the packet validity masks instead of pixel value, because real image pixels can be black.
masked_gray_stack = np.ma.array(gray_stack, mask=~valid_stack)
mean_image = masked_gray_stack.mean(axis=0).filled(0.0)
stdev_image = masked_gray_stack.std(axis=0).filled(0.0)

# Expand the histograms to have more contrast.
hist_mean_image = exposure.equalize_hist(mean_image)
hist_stdev_image = exposure.equalize_hist(stdev_image)

# Display the mean and standard deviation images.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(hist_mean_image, cmap='gray')
axes[0].set_title('Mean Image')
axes[0].axis('off')

axes[1].imshow(hist_stdev_image, cmap='gray')
axes[1].set_title('Standard Deviation Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()
